[<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>](https://colab.research.google.com/github/ADRIANVM117/quant-research-notes/blob/main/volatility_target/01_rebalancing_policies/02_rebalancing_policies.ipynb)

Abra la notebook en Google Colab y seleccione **Run all**. La serie diaria validada de SPY se descarga automáticamente desde este repositorio; no se requieren claves de API, Google Drive ni ejecutar notebooks anteriores.

# Volatility targeting con bandas

## Nueva etapa — Del pronóstico al rebalanceo


Nuestra pregunta sigue siendo:

> ¿Pueden las bandas de no negociación reducir los costos de volatility targeting sin deteriorar significativamente el control del riesgo?

La notebook contendrá Dos bloques, desarrollados progresivamente:

| Bloque | Contenido                                              |
| ------ | ------------------------------------------------------ |
| 03     | Exposición objetivo y políticas P0, P1 y P2            |
| 04     | Backtest con ejecución en apertura $ (t+1) $ y costos  |




Recordemos las tres políticas:

| Política         | Regla                                                         |
| ---------------- | ------------------------------------------------------------- |
| P0 — Daily       | Rebalanceo diario hacia el peso objetivo                      |
| P1 — Banda 5 pp  | Rebalanceo cuando la desviación supera 5 puntos porcentuales  |
| P2 — Banda 10 pp | Rebalanceo cuando la desviación supera 10 puntos porcentuales |

Mantendremos inicialmente:

- $(\lambda=0.94)$.
- Volatilidad objetivo del 12% anual.
- Exposición máxima de 100%.
- Capital inicial de 200,000 MXN como escenario.
- Señal al cierre de $(t)$ y ejecución en apertura de $(t+1)$.
- Un benchmark de exposición constante.
- Costos explícitos de implementación.

Hay un problema interesante que debemos resolver antes de programar el backtest completo.

Si calculamos el peso objetivo al cierre, pero operamos en la apertura siguiente, el mercado puede modificar la exposición efectiva durante la noche.

Eso significa que nuestra política deberá distinguir:

1. El peso deseado calculado al cierre.
2. El peso efectivo antes de operar en la siguiente apertura.
3. La diferencia entre ambos.
4. La decisión de operar o permanecer dentro de la banda.

Y al combinar retornos ajustados por dividendos con precios de apertura sin ajustar, tendremos que contabilizar correctamente los dividendos y los eventos corporativos para no distorsionar el valor económico de la cartera.

Este problema ya no es únicamente estadístico: es un problema de diseño de una política implementable.


In [1]:
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlretrieve
import importlib.util
import subprocess
import sys

missing_packages = []
if importlib.util.find_spec('polars') is None:
    missing_packages.append('polars==1.44.2')
if importlib.util.find_spec('plotly') is None:
    missing_packages.append('plotly==6.2.0')
if missing_packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *missing_packages])

import polars as pl

SPY_URL = (
    'https://raw.githubusercontent.com/ADRIANVM117/quant-research-notes/'
    'main/volatility_target/series/SPY.parquet'
)

def find_local_spy() -> Path | None:
    for folder in (Path.cwd(), *Path.cwd().parents):
        for candidate in (folder, folder / 'volatility_target'):
            path = candidate / 'series' / 'SPY.parquet'
            if path.is_file():
                return path
    return None

spy_path = find_local_spy()
if spy_path is None:
    spy_path = Path.cwd() / 'series' / 'SPY.parquet'
    temporary_path = spy_path.with_suffix('.part')
    spy_path.parent.mkdir(parents=True, exist_ok=True)
    print('Descargando SPY.parquet desde GitHub...')
    try:
        urlretrieve(SPY_URL, temporary_path)
        temporary_path.replace(spy_path)
    except URLError as error:
        raise RuntimeError('No fue posible descargar SPY.parquet desde GitHub.') from error

spy = pl.read_parquet(spy_path)
required_columns = {'session_date', 'open', 'high', 'low', 'close', 'adjusted_close'}
if spy.is_empty() or required_columns.difference(spy.columns):
    raise ValueError('SPY.parquet no cumple el contrato esperado de datos diarios.')

print(f'Filas: {spy.height:,}')
print(f"Periodo: {spy['session_date'].min()} a {spy['session_date'].max()}")

Filas: 2,227
Periodo: 2018-01-02 a 2026-11-10


In [2]:
import math

LAMBDA = 0.94
WARMUP_RETURNS = 20
TRADING_DAYS = 252

# El retorno r_t está disponible al cierre de la fecha t.
returns = (
    spy.select(
        'session_date',
        pl.col('adjusted_close').pct_change().alias('adjusted_return'),
    )
    .drop_nulls()
)

return_dates = returns['session_date'].to_list()
return_values = returns['adjusted_return'].to_list()
forecast_variance = [None] * len(return_values)

# La primera varianza usa sólo los primeros 20 retornos disponibles.
forecast_variance[WARMUP_RETURNS - 1] = sum(
    value ** 2 for value in return_values[:WARMUP_RETURNS]
) / WARMUP_RETURNS

# Cada fila t emite el pronóstico para la siguiente sesión.
for index in range(WARMUP_RETURNS, len(return_values)):
    forecast_variance[index] = (
        LAMBDA * forecast_variance[index - 1]
        + (1 - LAMBDA) * return_values[index] ** 2
    )

ewma = (
    pl.DataFrame({
        'signal_date': return_dates,
        'forecast_for_date': return_dates[1:] + [None],
        'adjusted_return': return_values,
        'forecast_variance': forecast_variance,
    })
    .with_columns(
        pl.col('forecast_variance').sqrt().alias('forecast_vol_daily'),
        (pl.col('forecast_variance').sqrt() * math.sqrt(TRADING_DAYS)).alias(
            'forecast_vol_annualized'
        ),
    )
    .filter(pl.col('forecast_variance').is_not_null())
    .filter(pl.col('forecast_for_date').is_not_null())
)

print(f'Pronósticos EWMA disponibles: {ewma.height:,}')
ewma.head()

Pronósticos EWMA disponibles: 2,206


signal_date,forecast_for_date,adjusted_return,forecast_variance,forecast_vol_daily,forecast_vol_annualized
date,date,f64,f64,f64,f64
2018-01-31,2018-02-01,0.000497,0.000034,0.00587,0.093187
2018-02-01,2018-02-02,-0.001135,0.000032,0.005698,0.090455
2018-02-02,2018-02-05,-0.02177,0.000059,0.007678,0.12189
2018-02-05,2018-02-06,-0.041822,0.00016,0.012664,0.201028
2018-02-06,2018-02-07,0.019702,0.000174,0.013192,0.20942



## Paso 02 - Cronologia operativa causal

La senal y la orden se separan explicitamente. El cierre de \(t\) es el ultimo instante en el que se puede decidir la orden destinada a la apertura de \(t+1\).

1. **Cierre de \(t\).** Observamos el retorno, actualizamos EWMA y calculamos \(w^*_{t+1|t}\).
2. **Cierre de \(t\), decision.** Valoramos las acciones al cierre bruto, junto con efectivo y receivables; P0/P1/P2 evaluan su banda con ese peso efectivo.
3. **Cierre de \(t\), orden.** Determinamos lado y cantidad. Las compras reciben un limite LOO y reservan efectivo y comision al peor precio permitido; las ventas reciben una cantidad MOO.
4. **Apertura de \(t+1\).** La compra no llena si `open > limit`; si llena, se aplica slippage adverso sin exceder el limite. La venta MOO usa la cantidad ya enviada.
5. **Resto de la sesion.** Valoramos al cierre y preparamos, con esa informacion, la orden de la sesion siguiente.

El peso que decide la banda es conocido al cierre:

$$
w_{t+1|t}^{-} = \frac{q_t C_t}{q_t C_t + c_t + r_t},
$$

con \(q_t\) acciones, \(c_t\) efectivo disponible y \(r_t\) dividendos por cobrar. `execution_open` no interviene en la decision ni en el dimensionamiento de la orden.


In [3]:
# Precios observables en cada extremo de la decisión.
signal_closes = spy.select(
    pl.col('session_date').alias('signal_date'),
    pl.col('close').alias('signal_close'),
)
execution_opens = spy.select(
    pl.col('session_date').alias('forecast_for_date'),
    pl.col('open').alias('execution_open'),
)

decision_schedule = (
    ewma.select(
        'signal_date',
        'forecast_for_date',
        'forecast_vol_annualized',
    )
    .join(signal_closes, on='signal_date', how='inner')
    .join(execution_opens, on='forecast_for_date', how='inner')
    .with_columns(
        (pl.col('execution_open') / pl.col('signal_close') - 1).alias(
            'overnight_return'
        )
    )
    .sort('signal_date')
)

assert decision_schedule.select(
    (pl.col('forecast_for_date') > pl.col('signal_date')).all()
).item()

print(f'Pares señal-apertura disponibles: {decision_schedule.height:,}')
decision_schedule.head()

Pares señal-apertura disponibles: 2,206


signal_date,forecast_for_date,forecast_vol_annualized,signal_close,execution_open,overnight_return
date,date,f64,f64,f64,f64
2018-01-31,2018-02-01,0.093187,281.9,281.07,-0.002944
2018-02-01,2018-02-02,0.090455,281.58,280.08,-0.005327
2018-02-02,2018-02-05,0.12189,275.45,273.45,-0.007261
2018-02-05,2018-02-06,0.201028,263.93,259.94,-0.015118
2018-02-06,2018-02-07,0.20942,269.13,268.5,-0.002341


### Qué tenemos hasta aquí

Cada fila de `decision_schedule` conecta una señal EWMA calculada al cierre de \(t\) con la apertura ejecutable de \(t+1\).

El campo `overnight_return` mide el movimiento que ocurre entre ambos momentos. Todavía no es una ganancia o pérdida de cartera: sólo muestra por qué el peso deseado al cierre puede diferir del peso efectivo justo antes de operar.

Con esta cronología evitamos look-ahead y ya podemos construir el peso objetivo y las reglas P0, P1 y P2 sobre una secuencia implementable.

## Paso 03 — Peso objetivo

El pronóstico EWMA se convierte en una exposición deseada a SPY. Si el riesgo previsto sube, reducimos exposición; si baja, la aumentamos, sin superar 100%.

$$
w^*_{t+1|t} = \min\left(1, \frac{0.12}{\widehat{\sigma}_{t+1|t}}\right).
$$

El resto de la cartera permanece en efectivo. Por ejemplo, con una volatilidad pronosticada de 24%, el peso objetivo es 50%. La fórmula usa sólo la señal disponible al cierre de \(t\); todavía no decide si operamos ni calcula el peso efectivo.

In [4]:
TARGET_VOLATILITY = 0.12
MAX_EXPOSURE = 1.00

target_schedule = (
    decision_schedule
    .with_columns(
        pl.min_horizontal(
            pl.lit(MAX_EXPOSURE),
            pl.lit(TARGET_VOLATILITY) / pl.col('forecast_vol_annualized'),
        ).alias('target_weight')
    )
    .with_columns(
        (pl.lit(1.0) - pl.col('target_weight')).alias('cash_weight')
    )
)

assert target_schedule.select(
    ((pl.col('target_weight') >= 0) & (pl.col('target_weight') <= MAX_EXPOSURE)).all()
).item()

print(f"Peso objetivo medio: {target_schedule['target_weight'].mean():.1%}")
print(
    f"Sesiones con 100% en SPY: "
    f"{target_schedule.select((pl.col('target_weight') == MAX_EXPOSURE).mean()).item():.1%}"
)

target_schedule.select(
    'signal_date',
    'forecast_for_date',
    'forecast_vol_annualized',
    'target_weight',
    'cash_weight',
).head()

Peso objetivo medio: 79.9%
Sesiones con 100% en SPY: 34.5%


signal_date,forecast_for_date,forecast_vol_annualized,target_weight,cash_weight
date,date,f64,f64,f64
2018-01-31,2018-02-01,0.093187,1.0,0.0
2018-02-01,2018-02-02,0.090455,1.0,0.0
2018-02-02,2018-02-05,0.12189,0.984495,0.015505
2018-02-05,2018-02-06,0.201028,0.596931,0.403069
2018-02-06,2018-02-07,0.20942,0.57301,0.42699


### Interpretación

Con la muestra actual, el peso objetivo medio es cercano a 80% y la estrategia desea estar completamente expuesta a SPY en alrededor de un tercio de las sesiones. Cuando EWMA detecta más riesgo, el peso objetivo cae; su mínimo histórico es cercano a 15%.

Estos valores describen lo que la señal **quiere** mantener. Aún no representan la posición real de la cartera: el movimiento overnight, las bandas y los costos determinarán después si se llega a ese peso.


## Paso 04 - Estado de cartera y peso efectivo

Para decidir una banda seguimos acciones de SPY, efectivo disponible, dividendos por cobrar y NAV. El peso efectivo usado para la decision se calcula con el cierre bruto disponible:

$$
w_{t+1|t}^{-} = \frac{\text{acciones}_t \times C_t}{\text{acciones}_t \times C_t + \text{cash}_t + \text{dividend\_receivable}_t}.
$$

Usaremos estas convenciones iniciales:

- Capital inicial: 200,000 unidades monetarias. Con acciones fraccionarias, la moneda no cambia los resultados porcentuales.
- Efectivo: rendimiento de 0% por ahora.
- Dividendos: en ex-date se reconocen como derecho neto por cobrar; se vuelven efectivo utilizable solo despues de payment-date. `adjusted_close` se usa unicamente para la senal EWMA.

El ejemplo siguiente ilustra el cambio de valoracion overnight, pero el motor operativo evalua la banda y fija cantidad al cierre anterior, no con ese precio de apertura.


In [5]:
INITIAL_CAPITAL = 200_000.0
DIVIDEND_CALENDAR_FILENAME = 'spy_dividend_calendar.csv'
DIVIDEND_CALENDAR_URL = (
    'https://raw.githubusercontent.com/ADRIANVM117/quant-research-notes/'
    '280837ae0bf56571d732b745906e837cbc58f5cc/volatility_target/'
    '01_rebalancing_policies/spy_dividend_calendar.csv'
)
DIVIDEND_CALENDAR_SHA256 = (
    'a7104bf1f72e238197398a68ddeb81615fa22b967a0a2c425800b971df9d5315'
)

import hashlib
import sys
from io import BytesIO
from urllib.request import urlopen

def find_local_file(filename):
    for folder in (Path.cwd(), *Path.cwd().parents):
        candidate = folder / filename
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f'No se encontro {filename}.')

try:
    dividend_calendar_bytes = find_local_file(DIVIDEND_CALENDAR_FILENAME).read_bytes()
except FileNotFoundError:
    if 'google.colab' not in sys.modules:
        raise
    try:
        with urlopen(DIVIDEND_CALENDAR_URL, timeout=30) as response:
            dividend_calendar_bytes = response.read()
    except OSError as error:
        raise RuntimeError('No se pudo descargar el calendario de dividendos.') from error

if hashlib.sha256(dividend_calendar_bytes).hexdigest() != DIVIDEND_CALENDAR_SHA256:
    raise ValueError('El hash SHA-256 del calendario de dividendos no coincide.')

dividend_calendar = (
    pl.read_csv(BytesIO(dividend_calendar_bytes))
    .with_columns(
        pl.col('ex_dividend_date').str.to_date('%Y-%m-%d'),
        pl.col('payment_date').str.to_date('%Y-%m-%d'),
        pl.col('dividend_amount').cast(pl.Float64),
    )
    .sort('ex_dividend_date')
)
calendar_columns = {
    'ex_dividend_date', 'payment_date', 'dividend_amount', 'source'
}
if calendar_columns.difference(dividend_calendar.columns):
    raise ValueError('El calendario de dividendos no cumple el contrato esperado.')
if (
    dividend_calendar.height != 35
    or dividend_calendar['ex_dividend_date'].n_unique() != dividend_calendar.height
    or not (dividend_calendar['payment_date'] > dividend_calendar['ex_dividend_date']).all()
):
    raise ValueError('El calendario de dividendos tiene eventos invalidos o duplicados.')

operating_sessions = target_schedule.select('forecast_for_date')
local_dividend_events = (
    spy.select(
        pl.col('session_date').alias('ex_dividend_date'),
        pl.col('dividend_amount').alias('local_dividend_amount'),
    )
    .filter(pl.col('local_dividend_amount') > 0)
    .join(
        operating_sessions,
        left_on='ex_dividend_date',
        right_on='forecast_for_date',
        how='inner',
    )
)
reconciled_dividends = local_dividend_events.join(
    dividend_calendar, on='ex_dividend_date', how='inner'
)
if (
    local_dividend_events.height != 35
    or reconciled_dividends.height != 35
    or local_dividend_events.join(dividend_calendar, on='ex_dividend_date', how='anti').height
    or dividend_calendar.join(local_dividend_events, on='ex_dividend_date', how='anti').height
    or not reconciled_dividends.select(
        (pl.col('local_dividend_amount').round(4)
         == pl.col('dividend_amount').round(4)).all()
    ).item()
):
    raise ValueError('Los 35 dividendos locales no conciliaron con el calendario.')

dividend_schedule = (
    spy.select(
        pl.col('session_date').alias('forecast_for_date'),
        pl.col('dividend_amount').alias('dividend_per_share'),
    )
    .join(
        dividend_calendar.select(
            pl.col('ex_dividend_date').alias('forecast_for_date'),
            'payment_date',
        ),
        on='forecast_for_date',
        how='left',
    )
)

portfolio_inputs = (
    target_schedule
    .join(dividend_schedule, on='forecast_for_date', how='inner')
    .sort('forecast_for_date')
)

def mark_to_open(shares, cash, open_price, dividend_per_share=0.0):
    """Valora una cartera con el dividendo como derecho por cobrar."""
    dividend_receivable = shares * dividend_per_share
    equity_value = shares * open_price
    portfolio_value = equity_value + cash + dividend_receivable
    if portfolio_value <= 0:
        raise ValueError('El valor de cartera debe ser positivo.')
    return {
        'cash': cash,
        'dividend_receivable': dividend_receivable,
        'portfolio_value': portfolio_value,
        'effective_weight': equity_value / portfolio_value,
    }

# Elegimos la primera sesión con exposición parcial para observar el drift del peso.
first = (
    portfolio_inputs
    .with_row_index('input_index')
    .filter(pl.col('target_weight') < MAX_EXPOSURE)
    .row(0, named=True)
)
shares = INITIAL_CAPITAL * first['target_weight'] / first['execution_open']
cash = INITIAL_CAPITAL * first['cash_weight']
initial_state = mark_to_open(shares, cash, first['execution_open'])

# Si no se opera de nuevo, el siguiente precio de apertura altera el peso efectivo.
next_session = portfolio_inputs.row(first['input_index'] + 1, named=True)
next_state = mark_to_open(
    shares,
    cash,
    next_session['execution_open'],
    next_session['dividend_per_share'],
)

portfolio_state_example = pl.DataFrame([
    {
        'moment': 'apertura inicial, después de asignar el objetivo',
        'session_date': first['forecast_for_date'],
        'open_price': first['execution_open'],
        'target_weight': first['target_weight'],
        'effective_weight': initial_state['effective_weight'],
        'dividend_receivable': initial_state['dividend_receivable'],
        'portfolio_value': initial_state['portfolio_value'],
    },
    {
        'moment': 'siguiente apertura, antes de decidir',
        'session_date': next_session['forecast_for_date'],
        'open_price': next_session['execution_open'],
        'target_weight': next_session['target_weight'],
        'effective_weight': next_state['effective_weight'],
        'dividend_receivable': next_state['dividend_receivable'],
        'portfolio_value': next_state['portfolio_value'],
    },
]).with_columns(
    (pl.col('effective_weight') - pl.col('target_weight')).alias('weight_deviation')
)

portfolio_state_example

moment,session_date,open_price,target_weight,effective_weight,dividend_receivable,portfolio_value,weight_deviation
str,date,f64,f64,f64,f64,f64,f64
"""apertura inicial, después de a…",2018-02-05,273.45,0.984495,0.984495,0.0,200000.0,0.0
"""siguiente apertura, antes de d…",2018-02-06,259.94,0.596931,0.983703,0.0,190272.055573,0.386772


**Cómo leer el ejemplo.** En la primera fila, el peso efectivo coincide con el objetivo porque acabamos de asignar la cartera. En la segunda, no se ha operado: el cambio en la apertura y el nuevo pronóstico EWMA hacen que el peso efectivo (98.37%) quede muy por encima del objetivo (59.69%). Esa diferencia de 38.68 pp es la información que usará cada política para decidir si rebalancea.



## Paso 05 - Reglas de rebalanceo

Una banda no modifica el peso objetivo: solo decide al cierre si debe enviarse una orden para la apertura siguiente. Definimos la desviacion como \(d = |w_{t+1|t}^{-} - w^*_{t+1|t}|\).

- **P0 - Daily:** prepara una orden cada cierre si la cantidad resultante no es cero.
- **P1 - Banda 5 pp:** prepara una orden si \(d > 0.05\).
- **P2 - Banda 10 pp:** prepara una orden si \(d > 0.10\).

La desigualdad es estricta. La apertura posterior solo determina el fill o no-fill y el peso realmente alcanzado; no vuelve a decidir la banda ni la cantidad.


In [6]:
POLICIES = {
    'P0 — Daily': None,
    'P1 — Banda 5 pp': 0.05,
    'P2 — Banda 10 pp': 0.10,
}

def should_rebalance(effective_weight, target_weight, policy):
    """Indica si la política exige rebalancear en esta apertura."""
    if policy not in POLICIES:
        raise ValueError(f'Política no reconocida: {policy}')

    deviation = abs(effective_weight - target_weight)
    tolerance = POLICIES[policy]
    return True if tolerance is None else deviation > tolerance

# Comprobación pequeña de las tres reglas antes de integrarlas al backtest.
rule_examples = pl.DataFrame({
    'case': ['dentro de 5 pp', 'entre 5 y 10 pp', 'por encima de 10 pp'],
    'effective_weight': [0.76, 0.86, 0.92],
    'target_weight': [0.80, 0.80, 0.80],
}).with_columns(
    (pl.col('effective_weight') - pl.col('target_weight')).abs().alias('deviation')
)

for policy in POLICIES:
    rule_examples = rule_examples.with_columns(
        pl.struct(['effective_weight', 'target_weight']).map_elements(
            lambda row: should_rebalance(
                row['effective_weight'], row['target_weight'], policy
            ),
            return_dtype=pl.Boolean,
        ).alias(policy)
    )

rule_examples


case,effective_weight,target_weight,deviation,P0 — Daily,P1 — Banda 5 pp,P2 — Banda 10 pp
str,f64,f64,f64,bool,bool,bool
"""dentro de 5 pp""",0.76,0.8,0.04,true,false,false
"""entre 5 y 10 pp""",0.86,0.8,0.06,true,true,false
"""por encima de 10 pp""",0.92,0.8,0.12,true,true,true



## Paso 06 - Mecanica de una orden causal

La orden se prepara con informacion del cierre. Para una compra se usa:

$$
L_{t+1} = C_t\left(1 + \widehat{\sigma}_{t+1|t,\mathrm{daily}}\right).
$$

La cantidad se resuelve contra el limite \(L_{t+1}\), incluyendo la comision en el presupuesto reservado. Por ello una compra nunca necesita efectivo negativo aunque la apertura llegue al limite. Una venta fija su cantidad al cierre usando el mismo objetivo y una referencia adversa de slippage; su ejecucion posterior es MOO.

El objetivo es un peso planificado bajo el precio limite. El peso alcanzado se registra en la apertura y puede diferir por un no-fill, por una apertura mas favorable que el limite, por slippage o por costos.


In [7]:
def rebalance_to_target(shares, cash, open_price, target_weight):
    """Rebalancea una cartera al objetivo, sin costos de transacción."""
    if not 0 <= target_weight <= MAX_EXPOSURE:
        raise ValueError('El peso objetivo está fuera de los límites permitidos.')

    portfolio_value = shares * open_price + cash
    target_equity = target_weight * portfolio_value
    target_shares = target_equity / open_price
    trade_notional = (target_shares - shares) * open_price
    cash_after = portfolio_value - target_equity

    return {
        'shares': target_shares,
        'cash': cash_after,
        'portfolio_value': portfolio_value,
        'trade_notional': trade_notional,
        'effective_weight': target_equity / portfolio_value,
    }

# Rebalanceamos la segunda fila del ejemplo anterior.
rebalanced_state = rebalance_to_target(
    shares=shares,
    cash=next_state['cash'],
    open_price=next_session['execution_open'],
    target_weight=next_session['target_weight'],
)

order_example = pl.DataFrame([
    {
        'moment': 'antes de operar',
        'effective_weight': next_state['effective_weight'],
        'cash': next_state['cash'],
        'portfolio_value': next_state['portfolio_value'],
        'trade_notional': 0.0,
    },
    {
        'moment': 'después de rebalancear',
        'effective_weight': rebalanced_state['effective_weight'],
        'cash': rebalanced_state['cash'],
        'portfolio_value': rebalanced_state['portfolio_value'],
        'trade_notional': rebalanced_state['trade_notional'],
    },
])

assert math.isclose(
    rebalanced_state['effective_weight'], next_session['target_weight']
)
order_example


moment,effective_weight,cash,portfolio_value,trade_notional
str,f64,f64,f64,f64
"""antes de operar""",0.983703,3100.932387,190272.055573,0.0
"""después de rebalancear""",0.596931,76692.810689,190272.055573,-73591.878302


**Cómo leer el ejemplo.** Antes de operar, SPY representa 98.37% de la cartera, mientras el objetivo es 59.69%. La orden vende aproximadamente 73,592 unidades monetarias de SPY —por eso `trade_notional` es negativo— y eleva el efectivo. El valor total no cambia porque aún no hay costos; sólo cambia la composición hasta alcanzar el objetivo.



## Paso 07 - Supuestos de implementacion: aproximacion a IBKR

A partir de aqui buscamos una simulacion causal e implementable, no una reproduccion exacta de cada estado de cuenta. Los supuestos se mantienen visibles.

- **Moneda.** SPY cotiza en USD. El motor conserva acciones, efectivo, dividendos y comisiones en USD. Los 200,000 MXN son el escenario de capital inicial y se convierten a USD antes de operar.
- **Ordenes.** Las compras son Limit-on-Open (LOO) y las ventas Market-on-Open (MOO). Lado, cantidad y reserva se calculan al cierre anterior; la simulacion no afirma compatibilidad confirmada de ordenes fraccionarias OPG en IBKR.
- **Limite y fills.** El limite de compra es \(C_{t-1}(1 + \widehat{\sigma}_{daily})\), con \(k=1\) fijo y no optimizado. Si `open > limit` no hay fill; si `open <= limit` se supone fill completo como aproximacion optimista de la subasta. El fill con slippage nunca supera el limite.
- **Comision.** Para cada orden de SPY usamos `min(1% * notional_usd, max(USD 1.00, USD 0.005 * acciones))` e IVA opcional sobre esa comision.
- **Dividendos y efectivo.** Los dividendos se acreditan netos de la retencion configurable. Un receivable integra NAV pero no financia una compra ni genera interes hasta estar disponible como efectivo.

Las tarifas pueden cambiar y dependen de la entidad y plan. La aproximacion OHLC no verifica fills reales de IBKR, liquidacion, impuestos personales ni impacto de mercado.


In [8]:
from io import BytesIO
from urllib.error import URLError
from urllib.request import urlopen

INITIAL_CAPITAL_MXN = 200_000.0
FX_COMMISSION_BPS = 0.20
FX_MIN_COMMISSION_USD = 2.00

IBKR_COMMISSION_PER_SHARE_USD = 0.005
IBKR_MIN_COMMISSION_USD = 1.00
IBKR_MAX_COMMISSION_RATE = 0.01
SLIPPAGE_BPS = 1.0
IVA_ON_COMMISSION = 0.00
DIVIDEND_WITHHOLDING = 0.10

FRED_USDMXN_URL = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=DEXMXUS'

def load_mxn_per_usd():
    """Carga pesos mexicanos por dólar desde la serie pública DEXMXUS."""
    try:
        with urlopen(FRED_USDMXN_URL, timeout=30) as response:
            raw_fx = pl.read_csv(BytesIO(response.read()), null_values='.')
    except URLError as error:
        raise RuntimeError('No fue posible descargar DEXMXUS desde FRED.') from error

    date_column, rate_column = raw_fx.columns[:2]
    return (
        raw_fx
        .select(
            pl.col(date_column).cast(pl.Utf8).str.to_date('%Y-%m-%d').alias('session_date'),
            pl.col(rate_column).cast(pl.Float64, strict=False).alias('mxn_per_usd'),
        )
        .drop_nulls()
        .sort('session_date')
    )

fx_mxnusd = load_mxn_per_usd()
first_execution_date = portfolio_inputs.select(pl.col('forecast_for_date').min()).item()
initial_fx_row = (
    fx_mxnusd
    .filter(pl.col('session_date') <= first_execution_date)
    .tail(1)
)
if initial_fx_row.is_empty():
    raise ValueError('No hay tipo de cambio disponible para la fecha inicial.')

initial_mxn_per_usd = initial_fx_row.item(0, 'mxn_per_usd')
gross_initial_usd = INITIAL_CAPITAL_MXN / initial_mxn_per_usd
initial_fx_commission_usd = max(
    FX_MIN_COMMISSION_USD, gross_initial_usd * FX_COMMISSION_BPS / 10_000
)
INITIAL_CAPITAL_USD = gross_initial_usd - initial_fx_commission_usd

print(f'Inicio operativo: {first_execution_date}')
print(f'Tipo de cambio inicial: {initial_mxn_per_usd:.4f} MXN/USD')
print(f'Capital disponible: USD {INITIAL_CAPITAL_USD:,.2f}')


Inicio operativo: 2018-02-01
Tipo de cambio inicial: 18.3600 MXN/USD
Capital disponible: USD 10,891.25


In [9]:
def execution_price(open_price, trade_shares, slippage_bps=SLIPPAGE_BPS):
    """Aplica deslizamiento adverso al precio de apertura."""
    if trade_shares > 0:
        return open_price * (1 + slippage_bps / 10_000)
    if trade_shares < 0:
        return open_price * (1 - slippage_bps / 10_000)
    return open_price

def ibkr_stock_cost(trade_shares, fill_price, iva_rate=IVA_ON_COMMISSION):
    """Calcula comisión IBKR Pro Fixed e IVA opcional, ambos en USD."""
    shares = abs(trade_shares)
    notional = shares * fill_price
    if shares == 0:
        return {'notional': 0.0, 'commission': 0.0, 'iva': 0.0, 'total_cost': 0.0}

    commission = min(
        IBKR_MAX_COMMISSION_RATE * notional,
        max(IBKR_MIN_COMMISSION_USD, IBKR_COMMISSION_PER_SHARE_USD * shares),
    )
    iva = commission * iva_rate
    return {
        'notional': notional,
        'commission': commission,
        'iva': iva,
        'total_cost': commission + iva,
    }

# Ilustración con la venta necesaria en el ejemplo del Paso 06.
example_trade_shares = rebalanced_state['shares'] - shares
example_fill_price = execution_price(
    next_session['execution_open'], example_trade_shares
)
example_cost = ibkr_stock_cost(example_trade_shares, example_fill_price)

execution_cost_example = pl.DataFrame([
    {
        'side': 'sell' if example_trade_shares < 0 else 'buy',
        'shares': abs(example_trade_shares),
        'open_price': next_session['execution_open'],
        'fill_price': example_fill_price,
        **example_cost,
    }
])

assert example_fill_price < next_session['execution_open']
assert example_cost['total_cost'] >= IBKR_MIN_COMMISSION_USD
execution_cost_example


side,shares,open_price,fill_price,notional,commission,iva,total_cost
str,f64,f64,f64,f64,f64,f64,f64
"""sell""",283.111019,259.94,259.914006,73584.519114,1.415555,0.0,1.415555


**Cómo leer el ejemplo.** La orden es una venta, por lo que el precio de llenado queda ligeramente por debajo de la apertura debido al deslizamiento de 1 bp. La comisión se calcula aparte sobre las acciones negociadas; como `IVA_ON_COMMISSION = 0`, el IVA es cero. `total_cost` es el importe que el motor descontará del valor de cartera cuando ejecute esta orden.



## Paso 08 - Contrato del backtest causal

El motor separa preparacion y ejecucion; las tres politicas reciben los mismos datos y supuestos.

| Momento | Informacion o accion |
| --- | --- |
| Cierre de `t-1` | EWMA, `target_weight`, `signal_close`, efectivo disponible y receivables ya son conocidos. Se calcula el peso efectivo, se evalua P0/P1/P2, se fija lado/cantidad y se reserva efectivo de una compra. |
| Apertura de `t` | Se acredita interes del efectivo previo, se transfieren pagos de dividendos programados y se reconocen derechos en ex-date. Ninguno de esos importes modifica la orden ya preparada. |
| Ejecucion | Una compra LOO llena completa solo si `open <= limit`; una venta MOO ejecuta la cantidad preparada. Se aplican slippage, comision e IVA, sin compras por encima del limite. |
| Cierre de `t` | Se valua la cartera con `close` y se prepara la orden de `t+1`. |

El peso de decision usa el cierre bruto, efectivo y receivables. `execution_open` solo participa despues al evaluar el fill, calcular el precio de ejecucion y valorar el estado realmente alcanzado.

Cada fila guarda fecha de decision, orden solicitada, cantidad enviada y ejecutada, lado, no-fill, limite, precio de ejecucion, reserva de efectivo, pesos antes/despues y contabilidad de dividendos. Las invariantes verifican cronologia, efectivo no negativo, long-only, limite de compra y conservacion de NAV cuando un receivable se cobra.


In [10]:

def maximum_buy_shares(cash, limit_price):
    """Calcula la cantidad maxima financiable al limite, incluida comision."""
    if cash < 0 or limit_price <= 0:
        raise ValueError('El presupuesto y el precio limite deben ser positivos.')

    lower_shares = 0.0
    upper_shares = cash / limit_price
    for _ in range(100):
        candidate_shares = (lower_shares + upper_shares) / 2
        costs = ibkr_stock_cost(candidate_shares, limit_price)
        required_cash = candidate_shares * limit_price + costs['total_cost']
        if required_cash <= cash:
            lower_shares = candidate_shares
        else:
            upper_shares = candidate_shares
    return lower_shares


def prepare_causal_order(
    shares,
    cash,
    dividend_receivable,
    signal_close,
    target_weight,
    forecast_vol_annualized,
    policy,
):
    """Prepara al cierre una orden causal para la apertura siguiente."""
    if (
        shares < 0
        or cash < 0
        or dividend_receivable < 0
        or signal_close <= 0
        or forecast_vol_annualized < 0
        or not 0 <= target_weight <= MAX_EXPOSURE
    ):
        raise ValueError('El estado al cierre no permite preparar una orden.')

    sigma_daily = forecast_vol_annualized / math.sqrt(TRADING_DAYS)
    limit_price = signal_close * (1 + sigma_daily)
    portfolio_value = shares * signal_close + cash + dividend_receivable
    if portfolio_value <= 0:
        raise ValueError('El valor de cartera al cierre debe ser positivo.')

    effective_weight = shares * signal_close / portfolio_value
    rebalance_requested = should_rebalance(
        effective_weight, target_weight, policy
    )
    order = {
        'portfolio_value_decision': portfolio_value,
        'effective_weight_decision': effective_weight,
        'signal_close': signal_close,
        'sigma_daily': sigma_daily,
        'limit_price': limit_price,
        'cash_available_at_decision': cash,
        'rebalance_requested': rebalance_requested,
        'order_requested': False,
        'order_side': 'none',
        'order_quantity': 0.0,
        'cash_reserved': 0.0,
        'planned_weight_after': effective_weight,
    }
    if not rebalance_requested or math.isclose(
        effective_weight, target_weight, abs_tol=1e-12
    ):
        return order

    if target_weight > effective_weight:
        side = 'buy'
        lower_shares = shares
        upper_shares = shares + maximum_buy_shares(cash, limit_price)
        valuation_price = limit_price

        def state_for(shares_after):
            trade_shares = shares_after - shares
            fill_price = limit_price
            costs = ibkr_stock_cost(trade_shares, fill_price)
            cash_after = cash - trade_shares * fill_price - costs['total_cost']
            portfolio_after = (
                shares_after * valuation_price + cash_after + dividend_receivable
            )
            return {
                'shares': shares_after,
                'cash': cash_after,
                'effective_weight': shares_after * valuation_price / portfolio_after,
                'fill_price': fill_price,
                **costs,
            }
    else:
        side = 'sell'
        lower_shares = 0.0
        upper_shares = shares
        valuation_price = signal_close

        def state_for(shares_after):
            trade_shares = shares_after - shares
            fill_price = execution_price(signal_close, trade_shares)
            costs = ibkr_stock_cost(trade_shares, fill_price)
            cash_after = cash - trade_shares * fill_price - costs['total_cost']
            portfolio_after = (
                shares_after * valuation_price + cash_after + dividend_receivable
            )
            return {
                'shares': shares_after,
                'cash': cash_after,
                'effective_weight': shares_after * valuation_price / portfolio_after,
                'fill_price': fill_price,
                **costs,
            }

    for _ in range(100):
        midpoint = (lower_shares + upper_shares) / 2
        if state_for(midpoint)['effective_weight'] < target_weight:
            lower_shares = midpoint
        else:
            upper_shares = midpoint

    shares_after = (lower_shares + upper_shares) / 2
    order_quantity = shares_after - shares
    if side == 'buy':
        order_quantity = max(0.0, order_quantity)
    else:
        order_quantity = min(0.0, max(order_quantity, -shares))

    planned_state = state_for(shares + order_quantity)
    cash_reserved = (
        order_quantity * limit_price + planned_state['total_cost']
        if side == 'buy' else 0.0
    )
    if cash_reserved > cash + 1e-8:
        raise RuntimeError('La reserva de compra excede el efectivo disponible.')

    return {
        **order,
        'order_requested': abs(order_quantity) > 1e-10,
        'order_side': side if abs(order_quantity) > 1e-10 else 'none',
        'order_quantity': order_quantity,
        'cash_reserved': cash_reserved,
        'planned_weight_after': planned_state['effective_weight'],
    }


def execute_causal_order(
    order, shares, cash, dividend_receivable, execution_open
):
    """Ejecuta en apertura una orden ya preparada al cierre anterior."""
    if execution_open <= 0:
        raise ValueError('La apertura debe ser positiva.')
    if order['order_quantity'] > 0 and cash + 1e-8 < order['cash_available_at_decision']:
        raise RuntimeError('El efectivo de apertura es menor al presupuesto reservado.')

    sent_shares = order['order_quantity'] if order['order_requested'] else 0.0
    executed_shares = 0.0
    fill_price = None
    no_fill = False

    if sent_shares > 1e-10:
        if execution_open > order['limit_price']:
            no_fill = True
        else:
            executed_shares = sent_shares
            fill_price = min(
                execution_price(execution_open, executed_shares),
                order['limit_price'],
            )
            if fill_price > order['limit_price'] + 1e-12:
                raise RuntimeError('Una compra LOO supero su precio limite.')
    elif sent_shares < -1e-10:
        executed_shares = max(sent_shares, -shares)
        fill_price = execution_price(execution_open, executed_shares)

    costs = ibkr_stock_cost(
        executed_shares,
        fill_price if fill_price is not None else execution_open,
    )
    shares_after = shares + executed_shares
    cash_after = (
        cash
        - executed_shares * (fill_price if fill_price is not None else execution_open)
        - costs['total_cost']
    )
    if cash_after < -1e-8:
        raise RuntimeError('La ejecucion requeriria efectivo negativo.')

    portfolio_value = (
        shares_after * execution_open + cash_after + dividend_receivable
    )
    return {
        'shares': shares_after,
        'cash': cash_after,
        'portfolio_value': portfolio_value,
        'effective_weight': (
            shares_after * execution_open / portfolio_value
            if portfolio_value > 0 else 0.0
        ),
        'trade_shares': executed_shares,
        'fill_price': fill_price,
        'no_fill': no_fill,
        'trade_notional': (
            executed_shares * fill_price if fill_price is not None else 0.0
        ),
        **costs,
    }


initial_order = prepare_causal_order(
    shares=0.0,
    cash=INITIAL_CAPITAL_USD,
    dividend_receivable=0.0,
    signal_close=first['signal_close'],
    target_weight=first['target_weight'],
    forecast_vol_annualized=first['forecast_vol_annualized'],
    policy=next(iter(POLICIES)),
)
causal_order_example = pl.DataFrame([{
    'signal_close': initial_order['signal_close'],
    'limit_price': initial_order['limit_price'],
    'order_side': initial_order['order_side'],
    'shares_sent': initial_order['order_quantity'],
    'cash_available_at_decision': initial_order['cash_available_at_decision'],
    'cash_reserved': initial_order['cash_reserved'],
    'planned_weight_after': initial_order['planned_weight_after'],
}])

assert initial_order['cash_reserved'] <= INITIAL_CAPITAL_USD + 1e-8
causal_order_example


signal_close,limit_price,order_side,shares_sent,cash_available_at_decision,cash_reserved,planned_weight_after
f64,f64,str,f64,f64,f64,f64
275.45,277.564999,"""buy""",38.626616,10891.246187,10722.396602,0.984495



**Como leer el ejemplo.** La tabla muestra una orden preparada al cierre: precio de cierre usado, limite LOO, lado, acciones enviadas, efectivo conocido y reserva maxima. No usa `execution_open` para decidir ni dimensionar; el fill solo se conocera al abrir.



## Paso 09 - Insumos diarios y rendimiento del efectivo

El motor necesita una fila por sesion con la senal del cierre previo, `signal_close`, `open`, `close`, dividendo y tasa del efectivo. `signal_close` y EWMA preparan la orden; `open` solo permite decidir el fill y valorar la apertura; `close` inicia la preparacion de la sesion posterior.

Usamos DFF de FRED como referencia historica diaria del efectivo en USD con los mismos supuestos de IBKR Pro: primeros USD 10,000 sin interes, excedente a `max(0, DFF - 0.50 pp)` y escala de NAV hasta USD 100,000. La tasa remunera unicamente efectivo, no receivables.

En ex-date se reconoce el dividendo neto con las acciones preexistentes. El receivable se transfiere a efectivo en la primera sesion estrictamente posterior a `payment_date`, conservando NAV. Si ese cobro ocurre en la apertura siguiente, sigue fuera del efectivo usado para preparar la orden en el cierre anterior.


In [11]:
IBKR_CASH_THRESHOLD_USD = 10_000.0
IBKR_CASH_FULL_NAV_USD = 100_000.0
IBKR_CASH_SPREAD = 0.005
FRED_DFF_URL = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=DFF'

def load_fred_dff():
    """Carga DFF como tasa anual porcentual."""
    try:
        with urlopen(FRED_DFF_URL, timeout=30) as response:
            raw_dff = pl.read_csv(BytesIO(response.read()), null_values='.')
    except URLError as error:
        raise RuntimeError('No fue posible descargar DFF desde FRED.') from error

    date_column, rate_column = raw_dff.columns[:2]
    return (
        raw_dff
        .select(
            pl.col(date_column).cast(pl.Utf8).str.to_date('%Y-%m-%d').alias('session_date'),
            pl.col(rate_column).cast(pl.Float64, strict=False).alias('dff_percent'),
        )
        .drop_nulls()
        .sort('session_date')
    )

dff = load_fred_dff()

# El motor verá la tasa de la sesión previa, nunca la observación contemporánea.
dff_for_sessions = (
    portfolio_inputs
    .select('forecast_for_date')
    .sort('forecast_for_date')
    .join_asof(
        dff,
        left_on='forecast_for_date',
        right_on='session_date',
        strategy='backward',
    )
    .with_columns(
        pl.col('dff_percent').shift(1).fill_null(0.0).alias('dff_percent_available')
    )
    .select('forecast_for_date', 'dff_percent_available')
)

closing_prices = spy.select(
    pl.col('session_date').alias('forecast_for_date'),
    pl.col('close').alias('close_price'),
)

backtest_inputs = (
    portfolio_inputs
    .join(closing_prices, on='forecast_for_date', how='inner')
    .join(dff_for_sessions, on='forecast_for_date', how='inner')
    .sort('forecast_for_date')
)

required_backtest_columns = {
    'signal_date', 'forecast_for_date', 'signal_close', 'execution_open', 'close_price',
    'target_weight', 'forecast_vol_annualized', 'dividend_per_share', 'payment_date',
    'dff_percent_available',
}
if required_backtest_columns.difference(backtest_inputs.columns):
    raise ValueError('Faltan columnas necesarias para el backtest.')
if backtest_inputs.drop('payment_date').null_count().select(
    pl.sum_horizontal(pl.all())
).item() != 0:
    raise ValueError('La tabla diaria contiene valores faltantes.')

execution_session_dates = backtest_inputs['forecast_for_date'].to_list()
payment_available_by_ex_date = {}
for event in dividend_calendar.iter_rows(named=True):
    available_date = next(
        (
            session_date for session_date in execution_session_dates
            if session_date > event['payment_date']
        ),
        None,
    )
    if available_date is None:
        raise ValueError('No hay sesion posterior a una fecha de pago.')
    payment_available_by_ex_date[event['ex_dividend_date']] = available_date
if len(payment_available_by_ex_date) != dividend_calendar.height:
    raise ValueError('No se pudo calendarizar todos los pagos de dividendos.')

print(f'Sesiones listas para el motor: {backtest_inputs.height:,}')
print(f'Periodo: {backtest_inputs["forecast_for_date"].min()} a {backtest_inputs["forecast_for_date"].max()}')
backtest_inputs.head(3)


Sesiones listas para el motor: 2,206
Periodo: 2018-02-01 a 2026-11-10


signal_date,forecast_for_date,forecast_vol_annualized,signal_close,execution_open,overnight_return,target_weight,cash_weight,dividend_per_share,payment_date,close_price,dff_percent_available
date,date,f64,f64,f64,f64,f64,f64,f64,date,f64,f64
2018-01-31,2018-02-01,0.093187,281.9,281.07,-0.002944,1.0,0.0,0.0,null,281.58,0.0
2018-02-01,2018-02-02,0.090455,281.58,280.08,-0.005327,1.0,0.0,0.0,null,275.45,1.42
2018-02-02,2018-02-05,0.12189,275.45,273.45,-0.007261,0.984495,0.015505,0.0,null,263.93,1.42


**Cómo leer el resultado.** Las 2,206 filas representan todas las sesiones en las que existe una señal EWMA ejecutable y los datos necesarios para contabilizar la cartera. La validación confirma que no hay huecos ni fechas adelantadas: cada objetivo proviene de un cierre anterior. Esta tabla todavía no contiene rendimientos de P0, P1 o P2; es el insumo común que recibirá cada política.


In [12]:
def cash_interest(cash, portfolio_value, dff_percent):
    """Calcula un día de interés sobre el efectivo elegible de IBKR."""
    eligible_cash = max(0.0, cash - IBKR_CASH_THRESHOLD_USD)
    nav_scale = min(1.0, portfolio_value / IBKR_CASH_FULL_NAV_USD)
    annual_rate = max(0.0, dff_percent / 100 - IBKR_CASH_SPREAD) * nav_scale
    daily_interest = eligible_cash * annual_rate / 360
    return {
        'eligible_cash': eligible_cash,
        'cash_rate_annual': annual_rate,
        'daily_interest': daily_interest,
    }

example_dff = backtest_inputs['dff_percent_available'][1]
cash_interest_example = pl.DataFrame([
    {
        'case': 'capital inicial',
        'cash': INITIAL_CAPITAL_USD,
        'portfolio_value': INITIAL_CAPITAL_USD,
        'dff_percent': example_dff,
        **cash_interest(INITIAL_CAPITAL_USD, INITIAL_CAPITAL_USD, example_dff),
    },
    {
        'case': 'NAV de USD 100,000',
        'cash': 20_000.0,
        'portfolio_value': 100_000.0,
        'dff_percent': example_dff,
        **cash_interest(20_000.0, 100_000.0, example_dff),
    },
])

assert cash_interest(5_000.0, 20_000.0, example_dff)['daily_interest'] == 0.0
cash_interest_example


case,cash,portfolio_value,dff_percent,eligible_cash,cash_rate_annual,daily_interest
str,f64,f64,f64,f64,f64,f64
"""capital inicial""",10891.246187,10891.246187,1.42,891.246187,0.001002,0.002481
"""NAV de USD 100,000""",20000.0,100000.0,1.42,10000.0,0.0092,0.255556



## Paso 10 - Motor diario de las politicas

`run_policy` mantiene una orden preparada desde el cierre previo. La primera orden se prepara con el estado inicial y la primera senal disponible. Despues de cada cierre, el motor prepara la orden de la siguiente sesion con acciones, efectivo disponible y receivables que existen en ese instante.

Para cada apertura, el motor:

1. acredita interes solo sobre el efectivo que sobrevivio la noche;
2. transfiere a efectivo los dividendos ya disponibles, conservando NAV;
3. reconoce en ex-date el receivable con las acciones preexistentes;
4. ejecuta la orden ya preparada: compra LOO o venta MOO;
5. registra fill/no-fill, efectivo posterior, peso alcanzado y valoracion de cierre;
6. al cierre, evalua la senal siguiente y prepara la nueva orden.

`effective_weight_before` es el peso de cierre que activo la banda; `effective_weight_open_before` muestra la valoracion observada antes de ejecutar; `effective_weight_after` es el peso realmente alcanzado. Las compras reservan efectivo y comision al limite; los pagos de dividendos que se acreditan en la apertura no alteran la cantidad enviada.


In [13]:

from collections import defaultdict


def run_policy(inputs, policy):
    """Ejecuta ordenes causales preparadas al cierre previo."""
    if policy not in POLICIES:
        raise ValueError(f'Politica no reconocida: {policy}')

    rows = list(inputs.iter_rows(named=True))
    if not rows:
        raise ValueError('No hay sesiones para ejecutar la politica.')

    shares = 0.0
    cash = INITIAL_CAPITAL_USD
    dividend_receivable = 0.0
    pending_dividend_payments = defaultdict(float)
    records = []

    def prepare_for_session(row):
        order = prepare_causal_order(
            shares=shares,
            cash=cash,
            dividend_receivable=dividend_receivable,
            signal_close=row['signal_close'],
            target_weight=row['target_weight'],
            forecast_vol_annualized=row['forecast_vol_annualized'],
            policy=policy,
        )
        order['decision_date'] = row['signal_date']
        order['pending_dividend_payment_at_decision'] = (
            pending_dividend_payments.get(row['forecast_for_date'], 0.0)
        )
        return order

    planned_order = prepare_for_session(rows[0])

    for session_index, row in enumerate(rows):
        if planned_order['decision_date'] != row['signal_date']:
            raise RuntimeError('La orden no corresponde al cierre de senal.')
        if not planned_order['decision_date'] < row['forecast_for_date']:
            raise RuntimeError('La orden causal no precede a su apertura.')

        open_price = row['execution_open']

        # El interes remunera solo el efectivo que sobrevivio la noche anterior.
        opening_value_before_interest = (
            shares * open_price + cash + dividend_receivable
        )
        interest = (
            cash_interest(cash, opening_value_before_interest, row['dff_percent_available'])
            if session_index > 0 else {'daily_interest': 0.0}
        )
        cash += interest['daily_interest']

        # El efectivo de un pago se acredita ahora, pero no financio la orden
        # preparada al cierre anterior.
        dividend_payment = pending_dividend_payments.pop(
            row['forecast_for_date'], 0.0
        )
        value_before_payment = shares * open_price + cash + dividend_receivable
        cash += dividend_payment
        dividend_receivable -= dividend_payment
        value_after_payment = shares * open_price + cash + dividend_receivable
        if not math.isclose(
            value_before_payment, value_after_payment, abs_tol=1e-8
        ):
            raise RuntimeError('El cobro de dividendos no conserva NAV.')

        # El derecho en ex-date se determina con las acciones preexistentes.
        dividend_net = shares * row['dividend_per_share'] * (1 - DIVIDEND_WITHHOLDING)
        if dividend_net:
            payment_available_date = payment_available_by_ex_date.get(
                row['forecast_for_date']
            )
            if payment_available_date is None:
                raise RuntimeError('Falta una fecha de disponibilidad de dividendo.')
            dividend_receivable += dividend_net
            pending_dividend_payments[payment_available_date] += dividend_net

        value_open_before_execution = (
            shares * open_price + cash + dividend_receivable
        )
        effective_weight_open_before = (
            shares * open_price / value_open_before_execution
            if value_open_before_execution > 0 else 0.0
        )
        state = execute_causal_order(
            planned_order, shares, cash, dividend_receivable, open_price
        )
        trade_executed = abs(state['trade_shares']) > 1e-10
        shares = state['shares']
        cash = state['cash']
        portfolio_value_close = (
            shares * row['close_price'] + cash + dividend_receivable
        )
        effective_weight_close = (
            shares * row['close_price'] / portfolio_value_close
            if portfolio_value_close > 0 else 0.0
        )

        records.append({
            'policy': policy,
            'session_date': row['forecast_for_date'],
            'signal_date': row['signal_date'],
            'order_decision_date': planned_order['decision_date'],
            'target_weight': row['target_weight'],
            'effective_weight_before': planned_order['effective_weight_decision'],
            'effective_weight_open_before': effective_weight_open_before,
            'effective_weight_after': state['effective_weight'],
            'effective_weight_close': effective_weight_close,
            'rebalance_requested': planned_order['rebalance_requested'],
            'order_requested': planned_order['order_requested'],
            'order_side': planned_order['order_side'],
            'order_quantity': planned_order['order_quantity'],
            'executed_shares': state['trade_shares'],
            'trade_executed': trade_executed,
            'no_fill': state['no_fill'],
            'signal_close': planned_order['signal_close'],
            'sigma_daily': planned_order['sigma_daily'],
            'limit_price': planned_order['limit_price'],
            'cash_available_at_decision': planned_order['cash_available_at_decision'],
            'cash_reserved': planned_order['cash_reserved'],
            'pending_dividend_payment_at_decision': (
                planned_order['pending_dividend_payment_at_decision']
            ),
            'shares': shares,
            'cash': cash,
            'dividend_net': dividend_net,
            'dividend_payment': dividend_payment,
            'dividend_receivable': dividend_receivable,
            'cash_interest': interest['daily_interest'],
            'open_price': open_price,
            'fill_price': state['fill_price'],
            'close_price': row['close_price'],
            'trade_shares': state['trade_shares'],
            'trade_notional': state['trade_notional'],
            'turnover_notional': abs(state['trade_notional']),
            'commission': state['commission'],
            'iva': state['iva'],
            'total_cost': state['total_cost'],
            'portfolio_value_decision': planned_order['portfolio_value_decision'],
            'portfolio_value_open': state['portfolio_value'],
            'portfolio_value_close': portfolio_value_close,
        })

        if session_index + 1 < len(rows):
            next_row = rows[session_index + 1]
            if next_row['signal_date'] != row['forecast_for_date']:
                raise RuntimeError('La siguiente senal no coincide con el cierre actual.')
            planned_order = prepare_for_session(next_row)

    history = pl.DataFrame(records)
    if history.height != len(rows):
        raise RuntimeError('El motor no recorrio todas las sesiones.')
    if not (history['session_date'] > history['order_decision_date']).all():
        raise RuntimeError('Se detecto una violacion de cronologia causal.')
    if not (history['order_decision_date'] == history['signal_date']).all():
        raise RuntimeError('La decision no usa el cierre de senal esperado.')
    if (
        not (history['cash'] >= -1e-8).all()
        or not (history['shares'] >= 0).all()
        or not (history['dividend_receivable'] >= -1e-8).all()
    ):
        raise RuntimeError('Se detecto una posicion no permitida.')
    if not history['effective_weight_after'].is_between(0, MAX_EXPOSURE).all():
        raise RuntimeError('Se detecto un peso fuera de limites.')

    executed_buys = history.filter(
        (pl.col('order_side') == 'buy') & pl.col('trade_executed')
    )
    if (
        not executed_buys.is_empty()
        and not executed_buys.select(
            (pl.col('fill_price') <= pl.col('limit_price') + 1e-10).all()
        ).item()
    ):
        raise RuntimeError('Una compra se ejecuto por encima de su limite.')
    no_fills = history.filter(pl.col('no_fill'))
    if (
        not no_fills.is_empty()
        and not no_fills.select(
            (
                (pl.col('order_side') == 'buy')
                & pl.col('order_requested')
                & ~pl.col('trade_executed')
            ).all()
        ).item()
    ):
        raise RuntimeError('Se detecto un no-fill inconsistente.')
    if pending_dividend_payments or not math.isclose(
        dividend_receivable, 0.0, abs_tol=1e-8
    ):
        raise RuntimeError('Quedaron dividendos pendientes al final de la muestra.')
    return history


policy_histories = {policy: run_policy(backtest_inputs, policy) for policy in POLICIES}

# Validamos bandas, causalidad y reservas de efectivo.
assert policy_histories[next(iter(POLICIES))]['rebalance_requested'].all()
for policy, tolerance in POLICIES.items():
    history = policy_histories[policy]
    if tolerance is not None:
        expected = (
            history['effective_weight_before'] - history['target_weight']
        ).abs() > tolerance
        assert (history['rebalance_requested'] == expected).all()
    assert (history['cash_reserved'] <= history['cash_available_at_decision'] + 1e-8).all()

policy_operational_summary = pl.concat([
    history.select(
        pl.lit(policy).alias('policy'),
        pl.len().alias('sessions'),
        pl.col('order_requested').sum().alias('orders_submitted'),
        pl.col('trade_executed').sum().alias('trades'),
        pl.col('no_fill').sum().alias('no_fills'),
        pl.col('turnover_notional').sum().alias('turnover_notional'),
        pl.col('total_cost').sum().alias('total_cost'),
        pl.col('portfolio_value_close').last().alias('final_portfolio_value'),
    )
    for policy, history in policy_histories.items()
])

policy_operational_summary


policy,sessions,orders_submitted,trades,no_fills,turnover_notional,total_cost,final_portfolio_value
str,u32,u32,u32,u32,f64,f64,f64
"""P0 — Daily""",2206,1590,1547,43,452333.748438,1278.516548,20946.398125
"""P1 — Banda 5 pp""",2206,268,263,5,292521.569102,263.0,22024.794437
"""P2 — Banda 10 pp""",2206,116,115,1,221203.198831,115.0,21982.64015



**Como leer el resultado.** La tabla resume la implementacion causal. `orders_submitted` distingue ordenes enviadas de `trades` realmente ejecutados; `no_fills` cuenta compras LOO que no cruzaron su limite. `turnover_notional`, `total_cost` y `final_portfolio_value` usan solo fills realizados. Las validaciones internas comprueban cronologia, reserva de efectivo, long-only, limite de compra y dividendos.


## Paso 11 — Resultados operativos

Antes de medir rendimiento y riesgo, observaremos cómo se comporta cada política como proceso de implementación. Estas salidas responden si las bandas realmente reducen actividad y fricción, y a qué costo de alejarse temporalmente del objetivo.

Mostraremos cuatro elementos:

| Resultado | Qué permite observar |
| --- | --- |
| Peso objetivo vs. peso efectivo | Cuándo la cartera sigue el objetivo y cuándo una banda tolera desviaciones. |
| Operaciones por política | Con qué frecuencia P0, P1 y P2 modifican realmente la posición. |
| Turnover acumulado | Cuánto capital se negocia a lo largo del tiempo, independiente del rendimiento. |
| Costos acumulados | Cuánto valor se pierde por comisión e IVA bajo cada regla. |

La serie de pesos se mostrará en una ventana reciente para que pueda leerse; las demás métricas usarán toda la muestra. Estas gráficas describen la operación del motor y no prueban todavía que una política sea mejor: la respuesta económica y estadística se desarrollará en la notebook 03.


In [14]:
import plotly.graph_objects as go

RECENT_SESSIONS = 252
POLICY_COLORS = {
    'P0 — Daily': '#1f77b4',
    'P1 — Banda 5 pp': '#ff7f0e',
    'P2 — Banda 10 pp': '#2ca02c',
}

operational_summary = policy_operational_summary.with_columns(
    (pl.col('turnover_notional') / INITIAL_CAPITAL_USD).alias('turnover_multiple'),
    (pl.col('total_cost') / INITIAL_CAPITAL_USD).alias('total_cost_pct'),
)
operational_summary

# 1. Peso objetivo y peso efectivo antes de decidir, en la última ventana anual.
recent_target = policy_histories['P0 — Daily'].tail(RECENT_SESSIONS)
fig_weights = go.Figure()
fig_weights.add_trace(go.Scatter(
    x=recent_target['session_date'].to_list(),
    y=recent_target['target_weight'].to_list(),
    name='Objetivo',
    line=dict(color='#111111', dash='dash', width=2),
))
for policy, history in policy_histories.items():
    recent = history.tail(RECENT_SESSIONS)
    fig_weights.add_trace(go.Scatter(
        x=recent['session_date'].to_list(),
        y=recent['effective_weight_before'].to_list(),
        name=policy,
        line=dict(color=POLICY_COLORS[policy], width=1.5),
    ))
fig_weights.update_layout(
    title='Peso objetivo y peso efectivo al cierre de decision',
    yaxis=dict(title='Peso en SPY', tickformat='.0%', range=[0, 1]),
    xaxis_title='',
    height=430,
    legend_title='',
)
fig_weights.show()

# 2. Número de operaciones efectivamente ejecutadas.
fig_trades = go.Figure(go.Bar(
    x=operational_summary['policy'].to_list(),
    y=operational_summary['trades'].to_list(),
    marker_color=[POLICY_COLORS[p] for p in operational_summary['policy'].to_list()],
    text=operational_summary['trades'].to_list(),
    textposition='outside',
))
fig_trades.update_layout(
    title='Operaciones ejecutadas por política',
    yaxis_title='Número de operaciones',
    height=360,
    showlegend=False,
)
fig_trades.show()

# 3 y 4. Turnover y costos, normalizados por el capital inicial.
fig_turnover = go.Figure()
fig_costs = go.Figure()
for policy, history in policy_histories.items():
    cumulative = history.with_columns(
        (pl.col('turnover_notional').cum_sum() / INITIAL_CAPITAL_USD).alias('cumulative_turnover'),
        (pl.col('total_cost').cum_sum() / INITIAL_CAPITAL_USD).alias('cumulative_cost_pct'),
    )
    dates = cumulative['session_date'].to_list()
    fig_turnover.add_trace(go.Scatter(
        x=dates, y=cumulative['cumulative_turnover'].to_list(),
        name=policy, line=dict(color=POLICY_COLORS[policy]),
    ))
    fig_costs.add_trace(go.Scatter(
        x=dates, y=cumulative['cumulative_cost_pct'].to_list(),
        name=policy, line=dict(color=POLICY_COLORS[policy]),
    ))

fig_turnover.update_layout(
    title='Turnover acumulado / capital inicial',
    yaxis_title='Veces el capital inicial',
    xaxis_title='',
    height=400,
    legend_title='',
)
fig_costs.update_layout(
    title='Costos acumulados / capital inicial',
    yaxis=dict(title='% del capital inicial', tickformat='.2%'),
    xaxis_title='',
    height=400,
    legend_title='',
)
fig_turnover.show()
fig_costs.show()
